**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/du-2026"], check=True)
    os.chdir("/content/encore/entries/du-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Du, Lai, Hu & Ye (2026) on MC_Maze (DANDI:000139, data elsewhere)

Paper doi:10.64898/2026.08.04.742876. MC_Maze-medium is not one of the datasets this paper is listed on
in the registry; the claims file marks its claims `data: elsewhere` and names the file:
DANDI:000139 version **0.220113.0408**, `sub-Jenkins/sub-Jenkins_ses-medium_desc-train_behavior+ecephys.nwb`,
streamed. It is an input of this attempt (protocol, author step 1.4). See `02_link_development.ipynb`
for the full claims table.

The loader is a port of the release's `mc_maze.py` (`load_mc_maze`, template context, called with
`neural_lead_ms=80` by `run_primary_analyses.py _run_nlb`), in `du2026.py` (`load_maze_raw`,
`build_maze_dataset`); the estimators are the same ports used for MC_RTT. The last section runs the
release's own estimator code, fetched at commit b042fbe, next to the ports.

| Claim | Status |
|---|---|
| maze-gain | tested |
| rff-maze | tested |
| estimator-family-nlb | MC_Maze part reported here (the outcome rests on MC_RTT, `06_mc_rtt.ipynb`) |

In [ ]:
import resource, time
import numpy as np
import pandas as pd
import du2026 as d

T0 = time.time()
EXTRA = (1000, 2000, 3000, 4000)
maze_assets = d.list_assets(d.MAZE_ID, d.MAZE_VERSION)
print([(a["path"], a["size"]) for a in maze_assets])
train_asset = next(a for a in maze_assets if a["path"] == d.MAZE_TRAIN_PATH)
raw = d.load_maze_raw(train_asset["asset_id"])
ds = d.build_maze_dataset(raw)

In [ ]:
print("behaviour series:", raw.descriptions)
tr = raw.trials
print(f"units: {raw.n_units} ({int(raw.heldout.sum())} marked heldout, all used, as mc_maze.py does); "
      f"spike counts per 20 ms bin {raw.spike_counts.min():.0f} to {raw.spike_counts.max():.0f}, mean rate {raw.spike_counts.mean() / 0.02:.1f} Hz")
print("trials:", len(tr), "split", tr.split.value_counts().to_dict(), "success", tr.success.value_counts().to_dict())
conds = tr.groupby(["trial_type", "trial_version"]).size()
print(f"conditions (trial_type, trial_version): {len(conds)}; trials per condition {conds.min()} to {conds.max()}")
val_counts = pd.Series(ds.condition[ds.test_indices, 0]).value_counts()
print(f"validation trials per condition: {val_counts.min()} to {val_counts.max()} (replacement needs at least 2)")
after = (tr.stop_time - tr.move_onset_time) * 1000
print(f"ms from movement onset to trial stop: {after.min():.0f} to {after.max():.0f}; trials with fewer than 800 ms: {int((after < 800).sum())}")
print(f"hand velocity range {np.nanmin(raw.hand_vel):.1f} to {np.nanmax(raw.hand_vel):.1f}; cursor {np.nanmin(raw.cursor_pos):.1f} to {np.nanmax(raw.cursor_pos):.1f}")
print(f"20 ms bins with no behaviour sample: {int((~np.isfinite(raw.hand_vel).all(axis=1)).sum())} of {raw.n_bins}")
print("analysis arrays: neural", ds.neural.shape, "context", ds.context.shape, "behaviour", ds.behavior.shape,
      "train/val", len(ds.train_indices), len(ds.test_indices))
print("missing (NaN) values in the analysis arrays: neural", int(np.isnan(ds.neural).sum()), "context", int(np.isnan(ds.context).sum()),
      "behaviour", int(np.isnan(ds.behavior).sum()))
# units and alignment: hand velocity (mm/s) against the derivative of cursor position (mm per 20 ms bin)
dcur = np.gradient(raw.cursor_pos, axis=0) / 0.02
for lag in (-2, 0, 2):
    a, b = raw.hand_vel[max(0, lag):len(dcur) + min(0, lag)], dcur[max(0, -lag):len(dcur) - max(0, lag)]
    ok = np.isfinite(a).all(axis=1) & np.isfinite(b).all(axis=1)
    print(f"corr(hand velocity, cursor velocity shifted {lag} bins): x {np.corrcoef(a[ok, 0], b[ok, 0])[0, 1]:.3f}, "
          f"y {np.corrcoef(a[ok, 1], b[ok, 1])[0, 1]:.3f}; SD ratio x {np.std(a[ok, 0]) / np.std(b[ok, 0]):.2f}")
# clock check: decode hand velocity from 100 ms of spikes ending `lag` bins before the behaviour bin, within trials
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
onsets = np.round(tr.move_onset_time.to_numpy() * 50).astype(int)
bins_ = np.concatenate([np.arange(o - 10, o + 40) for o in onsets])
bins_ = bins_[(bins_ > 40) & (bins_ < raw.n_bins - 40)]
bins_ = bins_[np.isfinite(raw.hand_vel[bins_]).all(axis=1)]
half = len(bins_) * 3 // 4
row = {}
for lag in (-10, -4, 0, 2, 4, 6, 10, 25):
    X = np.stack([raw.spike_counts[b - lag - 5:b - lag].reshape(-1) for b in bins_])
    m = make_pipeline(StandardScaler(), Ridge(alpha=1000.0)).fit(X[:half], raw.hand_vel[bins_[:half]])
    row[lag] = round(r2_score(raw.hand_vel[bins_[half:]], m.predict(X[half:]), multioutput="variance_weighted"), 3)
print("held-out R2 (last quarter of the trials) by lag in bins:", row)

## maze-gain

> In MC_Maze, a condition-by-phase template with geometry gave context-only R2 of 0.9053, and the fully
> nested neural gain beyond it was 0.00051 (conditional interval 0.00026 to 0.00076), a 0.54% MSE
> reduction (RMSE 81.76 to 81.53), with replacement gain -0.00004.

The criterion gives the data and the method (40 bins after measured movement onset, five spike bins
ending 80 ms before behaviour, 258-feature template context) and no rule. This attempt assumes the rule
of rtt-gain, the claim of the same Table 1 on the other NLB dataset: every expected value within its
tolerance. The outcome is marked as resting on this assumption. The replacement gain's tolerance
(0.000004) is narrower than the rounding of the paper's printed -0.00004 (-0.000045 to -0.000035);
this is asked of a reader (`questions/du-2026.yaml`), and the cell below also shows the value against
that rounding interval. Estimator seed 20260726 (`run_primary_analyses.py`);
the release split is fixed, and the estimator's folds and inner splits are also run under seeds plus
1000 to 4000. The conditional whole-trial interval is the release's (`_cluster_bootstrap_delta_r2`,
500 resamples, the script's default; the paper gives no count). The same fits give the three
nonlinear-neural maps (rff-maze).

In [ ]:
SEED = 20260726
jobs = [(f"nested_{SEED + k}", {}, "nested", dict(seed=SEED + k, rff_seeds=d.RFF_SEEDS, n_bootstrap=500 if k == 0 else 0), False)
        for k in (0,) + EXTRA]
jobs_family = [("block", {}, "block", {"seed": SEED}, False), ("rffctx", {}, "rff_context", {"seed": SEED}, False)]
t = time.time()
out = d.run_parallel(d.maze_jobs, [(train_asset["asset_id"], [j]) for j in jobs + jobs_family])
res = {k: v for o in out for k, v in o.items()}
print(f"fits in {(time.time() - t) / 60:.1f} min")
r = res[f"nested_{SEED}"]
exp = [("context-only R2", r["context_r2"], 0.9053, 0.09053), ("fully nested gain", r["gain"], 0.00051, 0.00025),
       ("% MSE reduction", r["mse_reduction_percent"], 0.54, 0.054), ("replacement gain", r["replacement_gain"], -0.00004, 0.000004)]
for label, v, e, tol in exp:
    print(f"{label}: {v:.7f} (expected {e} +- {tol}; within: {abs(v - e) <= tol})")
print(f"context RMSE {r['context_rmse']:.2f}, context-plus-neural RMSE {r['exact_rmse']:.2f} (paper 81.76 to 81.53)")
print(f"conditional interval {r['conditional_ci'][0]:.5f} to {r['conditional_ci'][1]:.5f} (paper 0.00026 to 0.00076)")
print("context features:", ds.context.shape[-1], "(258 in the criterion)")
for k in (0,) + EXTRA:
    rr = res[f"nested_{SEED + k}"]
    print(f"seed {SEED + k}: context R2 {rr['context_r2']:.4f}, gain {rr['gain']:.6f}, MSE reduction {rr['mse_reduction_percent']:.3f}%, "
          f"replacement {rr['replacement_gain']:.7f} (within the rounding of -0.00004: {-0.000045 <= rr['replacement_gain'] < -0.000035})")
print("penalties (context, residualizer, residual learner):", r["context_alpha"], r["residualizer_alpha"], r["innovation_alpha"])

**Result: reproduced (seed sensitive).** Under estimator seed 20260726 the context-only R2 is 0.90533, the fully nested gain 0.000513, the MSE reduction 0.542% (RMSE 81.76 to 81.53) and the replacement gain -0.0000371; all four are within tolerance, so the outcome is reproduced under the assumed rule (every value within its tolerance), and the conditional interval, 0.00026 to 0.00076, equals the paper's. The release's own code gives the same values on the same arrays (last section). The neural gain is small and depends on the estimator's folds: under seeds plus 1000 to 4000 the gain is -0.000138, 0.000479, 0.000528 and 0.000509, and the replacement gain -0.0027212, 0.0000294, -0.0000190 and 0.0000075, so the replacement gain is outside its tolerance, and outside the rounding of the paper's -0.00004, under every other seed, and changes sign under two; the gain is outside its tolerance under one.

## rff-maze

> Under the three nonlinear-neural maps, MC_Maze gain stayed practically zero (0.000045 to 0.000139).

Criterion as rff-rtt: D = 512 radial-basis random Fourier features, gamma = 1/p, map seeds 20260802 to
20260804 (`run_nonlinear_seed_sensitivity.py`), fully nested linear nuisances.

In [ ]:
g = {s: res[f"nested_{SEED}"]["rff"][s]["gain"] for s in d.RFF_SEEDS}
print({s: round(v, 7) for s, v in g.items()}, "penalties", {s: res[f"nested_{SEED}"]["rff"][s]["alpha"] for s in d.RFF_SEEDS})
lo, hi = min(g.values()), max(g.values())
print(f"lowest {lo:.7f} (expected 0.000045 +- 0.0000045; within: {abs(lo - 0.000045) <= 0.0000045}); "
      f"highest {hi:.7f} (expected 0.000139 +- 0.0000139; within: {abs(hi - 0.000139) <= 0.0000139})")
for k in EXTRA:
    gg = [res[f"nested_{SEED + k}"]["rff"][s]["gain"] for s in d.RFF_SEEDS]
    print(f"estimator seed {SEED + k}: map gains {min(gg):.7f} to {max(gg):.7f}")

**Result: reproduced (seed sensitive).** The three maps give MC_Maze gains of 0.0000448, 0.0000772 and 0.0001386, so the lowest and highest are within tolerance of the paper's 0.000045 and 0.000139. Under estimator seeds plus 1000 to 4000 the lowest of the three is 0.0000263 to 0.0000624, outside the tolerance under each, while every map gain stays positive and below 0.00014.

## estimator-family-nlb, MC_Maze part

The outcome of estimator-family-nlb rests on MC_RTT (`06_mc_rtt.ipynb`); the criterion reports the
MC_Maze part when the file is used, reading "near zero" as an absolute gain below 0.01
(`run_primary_analyses.py`, `maze_abs_gain_below_0_01`). Estimators as for MC_RTT: block penalties
and nonlinear context under the release's setting.

In [ ]:
print(f"block penalties: gain {res['block']['gain']:.6f} (penalties {res['block']['penalties']}); "
      f"nonlinear context (D 256, g chosen {res['rffctx']['gamma']}): gain {res['rffctx']['gain']:.6f}; fully nested linear {r['gain']:.6f}")
print("absolute MC_Maze gain below 0.01 under both families:", abs(res["block"]["gain"]) < 0.01 and abs(res["rffctx"]["gain"]) < 0.01)

MC_Maze part: the block-penalty gain is 0.000455 and the nonlinear-context gain -0.000243, both of absolute size below 0.01, so MC_Maze is near zero under both families on the criterion's reading. This part does not enter the outcome of estimator-family-nlb.

## The ports next to the release's own code

The verifier asked for one release function run next to its port. The release's estimator modules
(`types`, `estimators`, `specification_estimators`, `robustness_estimators`, `nested_estimators`) are
fetched at commit b042fbe and run on the same arrays: `evaluate_fully_nested_audit` and
`evaluate_nonlinear_neural_audit` on MC_Maze, and `evaluate_fully_nested_audit` on one LINK session
(2021-10-15 CO, SBP, geometry by phase, split seed 20260726). The loaders are the ports in both columns.

In [ ]:
from threadpoolctl import threadpool_limits
release, release_types = d.release_estimators()
rows = []
with threadpool_limits(1):
    rel = release.evaluate_fully_nested_audit(d.release_dataset(ds, release_types), seed=SEED, train_indices=ds.train_indices,
                                              test_indices=ds.test_indices, n_bootstrap=500)
    rows += [("MC_Maze", "context R2", r["context_r2"], rel.context_r2), ("MC_Maze", "gain", r["gain"], rel.exact_gain),
             ("MC_Maze", "replacement gain", r["replacement_gain"], rel.replacement_gain),
             ("MC_Maze", "interval low", r["conditional_ci"][0], rel.exact_gain_ci_low),
             ("MC_Maze", "interval high", r["conditional_ci"][1], rel.exact_gain_ci_high)]
    for s in d.RFF_SEEDS:
        nl = release.evaluate_nonlinear_neural_audit(d.release_dataset(ds, release_types), seed=SEED, rff_seed=s,
                                                     train_indices=ds.train_indices, test_indices=ds.test_indices)
        rows.append(("MC_Maze", f"RFF gain, map {s}", res[f"nested_{SEED}"]["rff"][s]["gain"], nl.nonlinear_gain))
    link_assets = d.list_assets(d.LINK_ID, d.LINK_VERSION)
    link_asset = next(a for a in link_assets if d.session_date(a["path"]) == "2021-10-15"
                      and d.link_task_from_metadata(d.asset_metadata(d.LINK_ID, d.LINK_VERSION, a["asset_id"])) == "CO")
    session = d.load_link_session(link_asset["asset_id"], link_asset["path"], signals=("SpikingBandPower",))
    lds = d.build_link_dataset(session, signal="SpikingBandPower", context_mode="geometry_tensor", seed=SEED)
    port_link = d.fully_nested(lds, seed=SEED)
    rel_link = release.evaluate_fully_nested_audit(d.release_dataset(lds, release_types), seed=SEED, train_indices=lds.train_indices,
                                                   test_indices=lds.test_indices, n_bootstrap=10)
    rows += [(f"LINK {session.date} {session.task}", "context R2", port_link["context_r2"], rel_link.context_r2),
             (f"LINK {session.date} {session.task}", "gain", port_link["gain"], rel_link.exact_gain),
             (f"LINK {session.date} {session.task}", "replacement gain", port_link["replacement_gain"], rel_link.replacement_gain)]
cmp_ = pd.DataFrame(rows, columns=["data", "quantity", "port", "release"])
cmp_["abs difference"] = (cmp_.port - cmp_.release).abs()
pd.set_option("display.float_format", lambda v: f"{v:.9f}")
print(cmp_.to_string())
print(f"largest absolute difference: {cmp_['abs difference'].max():.3g}")
print(f"run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

| Claim | Outcome |
|---|---|
| maze-gain | reproduced (seed sensitive) |
| rff-maze | reproduced (seed sensitive) |
| estimator-family-nlb | MC_Maze part holds (outcome in `06_mc_rtt.ipynb`) |

MC_Maze was read from DANDI:000139, which the paper uses but which is not listed as one of its
datasets here; the claims mark it as data elsewhere. Every value equals the paper's under estimator
seed 20260726. The gains are about 0.0005 and smaller, and the replacement gain and the lowest
nonlinear-map gain fall outside their 10% tolerances under the other estimator seeds. The release's
estimator code, run on the same arrays, gives the ported values exactly (largest absolute difference 0),
on MC_Maze and on one LINK session.

What was found about the file: 250 trials, all successful, 27 conditions with 8 to 10 trials each, 152
units (38 marked heldout) and the 188/62 split, as the paper states. Behaviour is stored only around
trials (600 of 38,644 20 ms bins have no sample), and the analysis arrays hold no missing value. The
behaviour series carry unit m/s or meters with conversion 0.001, so the stored values are mm/s and mm,
as their descriptions say. Velocity decoding from spikes peaks when the spikes end 2 to 4 bins (40 to
80 ms) before the behaviour bin.